## Start the vLLM Server.

VLLM_USE_TRITON_FLASH_ATTN=0 vllm serve Qwen/Qwen3-4B --served-model-name Qwen3-4B --api_key abc-123 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes --trust-remote-code --max_model_len 24272

## Install Dependencies

In [ ]:
!pip install nest_asyncio
!pip install langchain-openai

## Notebook Async Support

- Jupyter already has an event loop.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

print("Notebook Async Enabled!")

## vLLM Configuration

In [ ]:
import re
from typing import List
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

def get_llm(max_tokens: int = 8192):
    """
    Returns a ChatOpenAI client pointed at the local vLLM server.

    max_tokens is parameterized so individual agents can cap the
    output budget to match the expected size of their response
    (e.g. short remediation actions don't need an 8192 token budget).
    """

    return ChatOpenAI(
        model="Qwen3-4B",
        base_url="http://0.0.0.0:8000/v1",
        api_key="abc-123",
        temperature=0,
        max_tokens=max_tokens,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
        )


llm = get_llm()

print("Connected to vLLM!")


## Request & Domain Model

In [ ]:
#Request Models & Domain Models

class AlarmRequest(BaseModel):
    device: str
    event: str
    severity: str

class AlarmAnalysis(BaseModel):
    incident_type: str
    severity: str
    event: str

class LogAnalysis(BaseModel):
    interface_down: bool
    bgp_neighbor_down: bool
    route_withdrawal: bool
    findings: List[str]

## Alarm Agent & Log Agent

Purpose:
Parse router logs and extract findings.

In [ ]:
#Agents

class AlarmAgent:

    async def run(
            self,
            alarm: AlarmRequest
    ) -> AlarmAnalysis:
        
        incident_type = "Unknown"

        if alarm.event == "BGP_NEIGHBOR_DOWN":

            incident_type = (
                "Network Routing Failure"
            )

        return AlarmAnalysis(
            incident_type=incident_type,
            severity=alarm.severity,
            event=alarm.event
        )
    
class LogAgent:
    """
    Parse router logs and extract findings
    """

    INTERFACE_PATTERN = re.compile(
        r"interface.*down",
        re.IGNORECASE
    )

    BGP_PATTERN = re.compile(
        r"bgp.*down",
        re.IGNORECASE
    )

    ROUTE_PATTERN = re.compile(
        r"withdraw",
        re.IGNORECASE
    )

    async def run(self, logs: list[str]) -> LogAnalysis:

        findings = []

        interface_down = False
        bgp_neighbor_down = False
        route_withdrawal = False

        for line in logs:

            if self.INTERFACE_PATTERN.search(line):
                interface_down = True
                findings.append("Interface Down")

            if self.BGP_PATTERN.search(line):
                bgp_neighbor_down = True
                findings.append("BGP Neighbor Down")

            if self.ROUTE_PATTERN.search(line):
                route_withdrawal = True
                findings.append("Route Withdrawal")

        findings = list(set(findings))

        return LogAnalysis(
            interface_down=interface_down,
            bgp_neighbor_down=bgp_neighbor_down,
            route_withdrawal=route_withdrawal,
            findings=findings
        )

## Sample Alarm

In [ ]:
alarm = AlarmRequest(
    device="RTR-01",
    event="BGP_NEIGHBOR_DOWN",
    severity="CRITICAL"
)

alarm

## Sample Logs

In [ ]:
logs = [

    "Jul 10 10:15:01 RTR01 Interface GigabitEthernet0/0 down",

    "Jul 10 10:15:03 RTR01 LINK-3-UPDOWN",

    "Jul 10 10:15:04 RTR01 BGP-5-ADJCHANGE neighbor 192.168.1.1 Down",

    "Jul 10 10:15:05 RTR01 Route withdrawal initiated"
]

logs

## Execute Alarm Agent

In [ ]:
alarm_agent = AlarmAgent()

alarm_result = await alarm_agent.run(
    alarm
)

alarm_result

## Execute Log Agent

In [ ]:
log_agent = LogAgent()

log_result = await log_agent.run(
    logs
)

log_result

## Results Printing

In [ ]:
print("\n=== Alarm Analysis ===\n")

print(
    alarm_result.model_dump_json(
        indent=2
    )
)

print("\n=== Log Analysis ===\n")

print(
    log_result.model_dump_json(
        indent=2
    )
)

## Root Cause Analysis Model

In [ ]:
#RCA Models

from pydantic import BaseModel, Field

class RCAResult(BaseModel):

    root_cause: str = Field(description="Most probable root cause")
    impacted_component: str = Field(description="Affected Network Component")
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence score as a number between 0 and 1, e.g. 0.85"
    )
    reasoning: str = Field(description="Technical Reasoning")

#Correlation Helper: This is rule based correlation before invoking Qwen

class CorrelationEngine:

    def correlate(self, log_analysis: LogAnalysis) -> dict:

        scenario = "Unknown"
        probable_cause = "Unknown"

        if (log_analysis.interface_down and log_analysis.bgp_neighbor_down):
            scenario = ("BGP Failure caused by Interface Failure")

            probable_cause = ("WAN Interface Outage")

        elif log_analysis.bgp_neighbor_down:

            scenario = ("BGP Adjacency Loss")
            probable_cause = ("Routing Neighbor Failure")

        return {
            "scenario": scenario,
            "probable_cause":probable_cause,
            "findings":log_analysis.findings
        }


## RCA Agent

In [ ]:
class RCAAgent:

    def __init__(self):

        # method="function_calling" routes structured output through
        # vLLM's hermes tool-call parser - the mechanism the server was
        # actually started with (--enable-auto-tool-choice
        # --tool-call-parser hermes). The default method for ChatOpenAI
        # ("json_schema") instead relies on OpenAI-style constrained
        # decoding, which is what triggered the LengthFinishReasonError:
        # the small local model never emitted a stop token under that
        # path and burned through the full max_tokens budget.
        self.llm = (
            get_llm()
            .with_structured_output(
                RCAResult,
                method="function_calling"
            )
        )

    async def run(
        self,
        alarm_analysis: AlarmAnalysis,
        log_analysis: LogAnalysis
    ) -> RCAResult:

        correlation = (
            CorrelationEngine()
            .correlate(log_analysis)
        )

        prompt = f"""
You are a Senior Cisco TAC Engineer.

Analyze the incident.

Alarm Information

Incident Type:
{alarm_analysis.incident_type}

Event:
{alarm_analysis.event}

Severity:
{alarm_analysis.severity}

Correlation Results:
{correlation}

Log Findings:
{log_analysis.findings}

Determine:

1. Root Cause
2. Impacted Component
3. Confidence Score (a number between 0 and 1)
4. Technical Reasoning

Provide a short and concise RCA.
"""

        try:
            return await self.llm.ainvoke(prompt)

        except Exception as e:
            # Defensive fallback: never let an LLM hiccup take down the
            # whole pipeline. Degrade to the rule-based correlation result.
            print(
                f"[RCAAgent] Structured output failed "
                f"({type(e).__name__}); using rule-based fallback."
            )

            return RCAResult(
                root_cause=correlation.get("probable_cause", "Unknown"),
                impacted_component="Unknown - requires manual investigation",
                confidence=0.0,
                reasoning=(
                    "LLM structured output failed; falling back to "
                    "rule-based correlation only."
                )
            )


## Initiate RCA Agent

In [ ]:
rca_agent = RCAAgent()

print("RCA Agent Ready")

## Execute RCA Agent

In [ ]:
#Execute RCA Agent

rca_result = await rca_agent.run(
    alarm_result,
    log_result
)

rca_result

## Pretty Print RCA

In [ ]:
print(
    rca_result.model_dump_json(
        indent=2
    )
)

## Confidence Validation

- Sometimes models return values outside 0–1.

- Add a safeguard.

In [ ]:
#Confidence Validation

def normalize_confidence(confidence):
    """
    Safety net for RCAResult.confidence.

    RCAResult.confidence is now typed as a float with ge=0/le=1
    constraints, so in the normal path this is a no-op. It's kept
    as a defensive utility in case a model ever returns a text
    label ("High"/"Medium"/"Low") or a value outside the 0-1 range.
    """

    if isinstance(confidence, str):

        label_map = {"low": 0.0, "medium": 0.5, "high": 1.0}
        key = confidence.strip().lower()

        if key in label_map:
            return label_map[key]

        try:
            confidence = float(confidence)
        except ValueError:
            return 0.5

    return max(0.0, min(1.0, float(confidence)))


rca_result.confidence = (
    normalize_confidence(
        rca_result.confidence
    )
)

rca_result.confidence


## Execute RCA View
- Useful NOC Engineers

In [ ]:
# Execute RCA View : Useful NOC Engineers

print("\n=== ROOT CAUSE ANALYSIS ===\n")

print(
    f"Root Cause: "
    f"{rca_result.root_cause}"
)

print(
    f"Impacted Component: "
    f"{rca_result.impacted_component}"
)

print(
    f"Confidence: "
    f"{rca_result.confidence:.2f}"
)

print(
    f"Reasoning: "
    f"{rca_result.reasoning}"
)

## Remediation Model

In [ ]:
##Remediation Models

from pydantic import BaseModel, Field
from typing import List


class RemediationResult(BaseModel):

    actions: List[str] = Field(
        description="Recommended remediation actions", max_length=200
    )

    commands: List[str] = Field(
        description="Cisco CLI commands", max_length=200
    )

    validation_steps: List[str] = Field(
        description="Post-fix validation checks", max_length=200
    )


class IncidentSummary(BaseModel):

    incident_type: str

    root_cause: str

    confidence: float

    impacted_component: str

    summary: str


## CLI Knowledge Base

- For a POC, deterministic commands work better than asking the LLM to invent commands.

In [ ]:
class CiscoCommandLibrary:

    COMMANDS = {

        "WAN Interface Failure": {

            "commands": [

                "show interfaces GigabitEthernet0/0",

                "show interfaces status",

                "show logging",

                "show ip bgp summary",

                "show ip route",

                "shutdown",

                "no shutdown"
            ],

            "validation": [

                "Verify interface state is UP",

                "Verify BGP neighbor established",

                "Verify routes are present",

                "Verify alarms cleared"
            ]
        },

        "Routing Neighbor Failure": {

            "commands": [

                "show ip bgp summary",

                "show ip route",

                "ping neighbor_ip",

                "clear ip bgp *"
            ],

            "validation": [

                "Verify BGP session established",

                "Verify routing convergence"
            ]
        }
    }



## Remediation Agent

This agent combines:

- RCA result
- Deterministic Cisco knowledge
- LLM-generated action plan

In [ ]:


old_prompt = f"""
You are a Senior Cisco Network Engineer.

Root Cause:
{rca_result.root_cause}

Impacted Component:
{rca_result.impacted_component}

Generate short and concise remediation actions.

Return only actions.

Do not generate commands.
Commands already exist.
"""

In [ ]:
class RemediationAgent:

    def __init__(self):

        # Same fix as RCAAgent: method="function_calling" matches how
        # the vLLM server was started (--tool-call-parser hermes), and
        # avoids the LengthFinishReasonError seen with the default
        # "json_schema" method on this prompt. max_tokens is also capped
        # tighter here since the expected output (a handful of short
        # actions) is small - this makes any future runaway generation
        # fail fast and cheaply instead of burning 8192 tokens.
        self.llm = (
            get_llm(max_tokens=1024)
            .with_structured_output(
                RemediationResult,
                method="function_calling"
            )
        )

        # Plain (non-structured) LLM kept only as a last-resort
        # fallback if structured output still fails for any reason.
        self.raw_llm = get_llm(max_tokens=512)

    async def run(
        self,
        rca_result: RCAResult
    ) -> RemediationResult:

        kb = CiscoCommandLibrary()

        command_set = kb.COMMANDS.get(
            rca_result.root_cause,
            {}
        )

        prompt = f"""
You are a Senior Cisco Network Engineer.

Root Cause:
{rca_result.root_cause}

Impacted Component:
{rca_result.impacted_component}

Task:
Recommend up to 5 concise remediation actions (each under 15 words).
Focus only on actions - commands and validation steps are handled
separately and should not be included here.
"""

        try:
            llm_result = await self.llm.ainvoke(prompt)
            actions = llm_result.actions

        except Exception as e:
            print(
                f"[RemediationAgent] Structured output failed "
                f"({type(e).__name__}); using fallback parsing."
            )
            actions = await self._fallback_actions(prompt)

        return RemediationResult(

            actions=actions,

            commands=command_set.get(
                "commands",
                []
            ),

            validation_steps=command_set.get(
                "validation",
                []
            )
        )

    async def _fallback_actions(self, prompt: str) -> List[str]:
        """
        Last-resort path if structured output parsing fails
        (e.g. a truncated or malformed response). Asks for a plain
        text list instead and parses it manually.
        """

        try:
            raw = await self.raw_llm.ainvoke(
                prompt + "\n\nReturn ONLY a numbered list of actions, nothing else."
            )

            lines = [
                re.sub(r"^[\d\.\-\)\s]+", "", line).strip()
                for line in raw.content.splitlines()
                if line.strip()
            ]

            return lines[:5] if lines else ["Manual review required."]

        except Exception:
            return [
                "Manual review required - "
                "automatic remediation generation failed."
            ]


## Execute Remediation Agent


In [ ]:

remediation_agent = RemediationAgent()

remediation_result = await remediation_agent.run(
    rca_result
)

remediation_result

##Pretty Print Remediation


In [ ]:

print("\n=== REMEDIATION PLAN ===\n")

print("Actions")

for action in remediation_result.actions:

    print(f"- {action}")

print("\nCommands")

for command in remediation_result.commands:

    print(f"- {command}")

print("\nValidation")

for validation in remediation_result.validation_steps:

    print(f"- {validation}")

## Summary Agent

- Creates an executive-level report.
- No LLM required.
- This keeps the output deterministic.

In [ ]:
class SummaryAgent:

    async def run(
        self,
        alarm_analysis: AlarmAnalysis,
        rca_result: RCAResult,
        remediation_result: RemediationResult
    ) -> IncidentSummary:

        summary = f"""
A critical routing incident was detected.

Incident Type:
{alarm_analysis.incident_type}

Root Cause:
{rca_result.root_cause}

Impacted Component:
{rca_result.impacted_component}

Confidence:
{rca_result.confidence}

Technical Reasoning:
{rca_result.reasoning}

Recommended Actions:
{', '.join(remediation_result.actions)}
"""

        return IncidentSummary(

            incident_type=
            alarm_analysis.incident_type,

            root_cause=
            rca_result.root_cause,

            confidence=
            rca_result.confidence,

            impacted_component=
            rca_result.impacted_component,

            summary=summary
        )

## Execute Summary Agent

In [ ]:
summary_agent = SummaryAgent()

summary_result = await summary_agent.run(
    alarm_result,
    rca_result,
    remediation_result
)

summary_result


## Executive Report

In [ ]:
print(
    summary_result.summary
)

## Incident Dashboard View

- Useful for NOC Engineers


In [ ]:
dashboard = {

    "Incident Type":
    summary_result.incident_type,

    "Root Cause":
    summary_result.root_cause,

    "Confidence":
    summary_result.confidence,

    "Impacted Component":
    summary_result.impacted_component,

    "Actions":
    remediation_result.actions,

    "Commands":
    remediation_result.commands
}

dashboard

## Final Response Model
- This is the object returned to the NOC engineer.

In [ ]:
from pydantic import BaseModel
from typing import List


class IncidentReport(BaseModel):

    incident_type: str

    root_cause: str

    impacted_component: str

    confidence: float

    actions: List[str]

    commands: List[str]

    validation_steps: List[str]

    summary: str

## Incident Analyser
- This orchestrates the complete workflow.

In [ ]:
class IncidentAnalyzer:

    def __init__(self):

        self.alarm_agent = AlarmAgent()

        self.log_agent = LogAgent()

        self.rca_agent = RCAAgent()

        self.remediation_agent = (
            RemediationAgent()
        )

        self.summary_agent = (
            SummaryAgent()
        )

    async def analyze(
        self,
        alarm: AlarmRequest,
        logs: list[str]
    ) -> IncidentReport:

        print("Step 1: Alarm Analysis")

        alarm_result = (
            await self.alarm_agent.run(
                alarm
            )
        )

        print("Step 2: Log Analysis")

        log_result = (
            await self.log_agent.run(
                logs
            )
        )

        print("Step 3: Root Cause Analysis")

        rca_result = (
            await self.rca_agent.run(
                alarm_result,
                log_result
            )
        )

        print("Step 4: Remediation")

        remediation_result = (
            await self.remediation_agent.run(
                rca_result
            )
        )

        print("Step 5: Summary")

        summary_result = (
            await self.summary_agent.run(
                alarm_result,
                rca_result,
                remediation_result
            )
        )

        return IncidentReport(

            incident_type=
            alarm_result.incident_type,

            root_cause=
            rca_result.root_cause,

            impacted_component=
            rca_result.impacted_component,

            confidence=
            rca_result.confidence,

            actions=
            remediation_result.actions,

            commands=
            remediation_result.commands,

            validation_steps=
            remediation_result.validation_steps,

            summary=
            summary_result.summary
        )

## Create Analyser

In [ ]:
analyzer = IncidentAnalyzer()

print("Telecom NOC Copilot Ready")

## Sample Alarm

In [ ]:
alarm = AlarmRequest(

    device="RTR-01",

    event="BGP_NEIGHBOR_DOWN",

    severity="CRITICAL"
)

## Sample Logs

In [ ]:
logs = [

    "Jul 10 10:15:01 RTR01 Interface GigabitEthernet0/0 down",

    "Jul 10 10:15:03 RTR01 LINK-3-UPDOWN",

    "Jul 10 10:15:04 RTR01 BGP-5-ADJCHANGE neighbor 192.168.1.1 Down",

    "Jul 10 10:15:05 RTR01 Route withdrawal initiated"
]

## Execute End-to-End Workflow

In [ ]:
incident_report = await analyzer.analyze(
    alarm,
    logs
)

incident_report

## JSON Output

- Useful when integrating with APIs later.

In [ ]:
print(
    incident_report.model_dump_json(
        indent=2
    )
)

## NOC Dashboard View

- Engineers usually want a compact view.

In [ ]:
from pprint import pprint

dashboard = {

    "Incident":
    incident_report.incident_type,

    "Root Cause":
    incident_report.root_cause,

    "Confidence":
    f"{incident_report.confidence:.2f}",

    "Impacted Component":
    incident_report.impacted_component,

    "Recommended Actions":
    incident_report.actions,

    "CLI Commands":
    incident_report.commands
}

pprint(dashboard)

## Executive Summary

- For management or shift handover.

In [ ]:
print("\n========== INCIDENT REPORT ==========\n")

print(incident_report.summary)

print("\n=====================================\n")

## Test Another Scenario

- We can now simulate different incidents.

In [ ]:
alarm2 = AlarmRequest(
    device="RTR-02",
    event="BGP_NEIGHBOR_DOWN",
    severity="MAJOR"
)

logs2 = [

    "BGP neighbor 10.10.10.1 Down",

    "BGP keepalive timeout",

    "Route withdrawal initiated"
]

incident_report2 = await analyzer.analyze(
    alarm2,
    logs2
)

incident_report2